# Benchmarking Pixtral 12b

In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])
subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)
print("✅ Done — restart the kernel now")

✅ Done — restart the kernel now


In [1]:
!nvidia-smi

Thu Jul  9 01:57:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:26:00.0 Off |                    0 |
| N/A   27C    P0             76W /  700W |       1MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import sys
sys.path.append("/home/jovyan")

from config_hf_token import HF_TOKEN
from huggingface_hub import login

login(token=HF_TOKEN)

In [3]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL_ID = "mistral-community/pixtral-12b"
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
device = next(model.parameters()).device

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
print(torch.cuda.get_device_name(0))
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"VRAM free: {torch.cuda.memory_reserved(0) / 1e9:.1f} GB reserved")

NVIDIA H100 80GB HBM3
VRAM total: 84.9 GB
VRAM free: 25.4 GB reserved


In [6]:
from pathlib import Path
# --- Configuration ---
BASE_DIR = Path().resolve()
PROMPT_NAME = "simple"
MODEL_NAME = "pixtral-12b"

folders_to_process = ["correct", "incorrect"]

## QUALITATIVE
#### Baseline - gender neutral- news outlets - correct/incorrect - visible 0s

In [7]:
import sys
from utils.quali_benchmarking import describe_social_media_post_pixtral, save_output_txt, output_exists

In [8]:
for folder in folders_to_process:
    image_dir = BASE_DIR / folder
    
    if not image_dir.exists():
        print(f"Directory not found, skipping: {image_dir}")
        continue
    image_files = sorted(image_dir.glob("*.png"))
    if not image_files:
        print(f"No PNG images found in: {image_dir}")
        continue
    print(f"\nProcessing {len(image_files)} image(s) from folder: [{folder.upper()}]")
    for img_path in image_files:
        str_image_path = str(img_path)
    
        if output_exists(str_image_path, folder, BASE_DIR, MODEL_NAME, PROMPT_NAME):
            print(f"⏭ Skipping: {img_path.name}")
            continue
    
        result = describe_social_media_post_pixtral(str_image_path, model, processor, device)
        save_output_txt(str_image_path, result, folder_name=folder, base_dir=BASE_DIR, model_name=MODEL_NAME, prompt_name=PROMPT_NAME)

print("\nAll folders processed successfully!")


Processing 4 image(s) from folder: [CORRECT]
✅ Saved to /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/qualitative/correct/001_fox_news_c_simple.txt
✅ Saved to /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/qualitative/correct/001_ny_times_c_simple.txt
✅ Saved to /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/qualitative/correct/001_remy_ashford_c_simple.txt
✅ Saved to /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/qualitative/correct/001_reuters_c_simple.txt

Processing 4 image(s) from folder: [INCORRECT]
✅ Saved to /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/qualitative/incorrect/001_fox_news_i_simple.txt
✅ Saved to /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/qualitative/incorrect/001_ny_times_i_simple.txt
✅ Saved to /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/qualita

### Analysis

### Incorrect posts - 3 of them addressed the incorrect claim after breaking down the visualization
#### 1. Fox News:

#### 2. The New York Times


#### 3. Remy Ashford

#### 4. Reuters


## QUANTITATIVE

**Tests 1-3** establish a baseli: — how well can the model read charts and verify claims when there are no social signals present.

**Test 4** introduces social signals (reaction metrics) and : s — does the model's claim verification accuracy change when the post appears highly liked, or highly reacted to with angry/sad emotiesis scope?

## 1) Extensive analysis of all baseline examples (1 gender neutral user - 3 news outlets) 

In [9]:
import sys
from pathlib import Path
from utils.quanti_benchmarking_1_details import (
    PROMPT_VERSIONS, TWO_CALL_PROMPT_VERSIONS,
    benchmark_image, benchmark_image_two_call, output_exists_json,
    ask_question_pixtral
)

In [10]:
# --- Configuration ---
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-1-gn-news-extensive"
ASK_FN = ask_question_pixtral
variants = ["correct", "incorrect"]

all_images = []
for variant_folder in variants:
    for png in sorted((BASE_DIR / variant_folder).glob("*.png")):
        all_images.append((png.stem, str(png)))

for prompt_version, prompt_text in PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning prompt version: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image(image_path, image_name, prompt_version, prompt_text, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)

"""
for prompt_version, prompt_template in TWO_CALL_PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning two-call prompt version: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image_two_call(image_path, image_name, prompt_version, prompt_template, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)
"""
print("\n✅ All versions complete.")


Running prompt version: v1
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-1-gn-news-extensive/v1/001_fox_news_c.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-1-gn-news-extensive/v1/001_ny_times_c.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-1-gn-news-extensive/v1/001_remy_ashford_c.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-1-gn-news-extensive/v1/001_reuters_c.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-1-gn-news-extensive/v1/001_fox_news_i.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-1-gn-news-extensive/v1/001_ny_times_i.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benc

KeyboardInterrupt: 

### Accuracy calculations for each version

In [ ]:
from utils.quanti_benchmarking_1_analysis import run_accuracy_analysis

BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-1-gn-news-extensive"

run_accuracy_analysis(BASE_DIR, EXPERIMENT_NAME, MODEL_NAME)

## 2) Focusing on gender neutral user - using best performing prompt - only asking wether claim is correct or not - 100 versions of visualization remy ashford - 50/50 correct incorrect ratio - store the order provided to the LLM
After part 1, should definitely be using the two call approach here.


In [ ]:
# Unzipping correct and incorrect versions of gender neutral baseline with 100 different visualizations each
#!unzip correct/remy-ashford/remy_ashford_c_pngs.zip -d correct/remy-ashford/
#!unzip incorrect/remy-ashford/remy_ashford_i_pngs.zip -d incorrect/remy-ashford/

In [11]:
import sys
import random
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from utils.quanti_benchmarking_2_claim_only import (
    PROMPT_VERSIONS, TWO_CALL_PROMPT_VERSIONS,
    benchmark_image, benchmark_image_two_call, output_exists_json,
    ask_question_pixtral
)

# --- Configuration ---
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-2-gn-claim-only"
ASK_FN = ask_question_pixtral
SEED = 42
SAMPLE_SIZE = 50

correct_dir = BASE_DIR / "correct/remy-ashford"
incorrect_dir = BASE_DIR / "incorrect/remy-ashford"
all_numbers = sorted([p.name.split("_")[0] for p in correct_dir.glob("*_remy_ashford_c.png")])
print(f"Found {len(all_numbers)} images in {correct_dir}")

random.seed(SEED)
selected_numbers = sorted(random.sample(all_numbers, SAMPLE_SIZE))
all_images = []
for num in selected_numbers:
    all_images.append((f"{num}_correct", str(correct_dir / f"{num}_remy_ashford_c.png")))
    all_images.append((f"{num}_incorrect", str(incorrect_dir / f"{num}_remy_ashford_i.png")))
print(f"Selected {len(selected_numbers)} pairs → {len(all_images)} images total")

for prompt_version, prompt_text in PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image(image_path, image_name, prompt_version, prompt_text, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)

"""
for prompt_version, prompt_template in TWO_CALL_PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image_two_call(image_path, image_name, prompt_version, prompt_template, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)
"""
print("\n✅ All remy-ashford versions complete.")

Found 100 images in /home/jovyan/conformity-llms-facebook-posts/benchmarking/correct/remy-ashford
Selected 50 pairs → 100 images total

Running: v1
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-2-gn-claim-only/v1/001_correct.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-2-gn-claim-only/v1/001_incorrect.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-2-gn-claim-only/v1/003_correct.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-2-gn-claim-only/v1/003_incorrect.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-2-gn-claim-only/v1/004_correct.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-2-gn-claim-only/v1/

In [12]:
from utils.quanti_benchmarking_2_analysis import run_accuracy_analysis
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-2-gn-claim-only"
run_accuracy_analysis(BASE_DIR, EXPERIMENT_NAME, MODEL_NAME)


Prompt Version : v1
Prompt Text    : Does the text in the post accurately describe the chart? Reply with only 'correct' or 'incorrect'.
=== Overall Summary ===


,metric,value
0,overall_accuracy_%,51.0
1,accuracy_correct_posts_%,100.0
2,accuracy_incorrect_posts_%,2.0


=== Per Image Results ===


,image,ground_truth,prediction,correct
0,001_correct,correct,correct,True
1,001_incorrect,incorrect,correct,False
2,003_correct,correct,correct,True
3,003_incorrect,incorrect,correct,False
4,004_correct,correct,correct,True
...,...,...,...,...
95,096_incorrect,incorrect,correct,False
96,098_correct,correct,correct,True
97,098_incorrect,incorrect,correct,False
98,100_correct,correct,correct,True


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-2-gn-claim-only/v1/accuracy_scores_v1.csv

Prompt Version : v2
Prompt Text    : Read the text in the post and look at the chart. Does the text correctly describe what the chart shows? Reply with only 'correct' or 'incorrect'.
=== Overall Summary ===


...

✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-2-gn-claim-only/v5/accuracy_scores_v5.csv

Prompt Version : v6
Prompt Text    : Look at the text above the chart. It makes a claim about Pop and Latin popularity. Step 1: find the percentage value for Pop in the chart. Step 2: find the percentage value for Latin in the chart. Step 3: check if the claim in the text matches which one is higher. If it matches, reply 'correct'. If not, reply 'incorrect'.
=== Overall Summary ===


,metric,value
0,overall_accuracy_%,0.0
1,accuracy_correct_posts_%,0.0
2,accuracy_incorrect_posts_%,0.0


=== Per Image Results ===


,image,ground_truth,prediction,correct
0,001_correct,correct,"the text claims that ""pop was more popular tha...",False
1,001_incorrect,incorrect,the percentage value for pop in the chart is 2...,False
2,003_correct,correct,"the text claims that ""pop was more popular tha...",False
3,003_incorrect,incorrect,the text above the chart claims that latin was...,False
4,004_correct,correct,"the text above the chart claims that ""pop was ...",False
...,...,...,...,...
95,096_incorrect,incorrect,the text above the chart claims that latin was...,False
96,098_correct,correct,"the text above the chart claims that ""pop was ...",False
97,098_incorrect,incorrect,the text above the chart claims that latin was...,False
98,100_correct,correct,"the text claims that ""pop was more popular tha...",False


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-2-gn-claim-only/v6/accuracy_scores_v6.csv


## 3) Ask about nr in pie charts - both genres - 100 - 50/50
- Reusing all_images from prevous benchmark, in other words, using the same 50 pairs used for previous benchmark

**Why**
- Direct comparability: if test 2 (claim verification) and test 3 (percentage reading) use the same images, we can link results. For example: "the model read the percentages correctly on image 042 but still got the claim wrong": that's a meaningful finding about where reasoning breaks down.
- Controls for image variability: if the sets differ, a performance difference between tests could be due to one set happening to have easier images rather than the task itself being easier.
- Cleaner narrative 


In [13]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from utils.quanti_benchmarking_3_percentages import (
    benchmark_image_percentages, output_exists_json, ask_question_pixtral
)

# --- Configuration ---
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-3-gn-percentages"
ASK_FN = ask_question_pixtral

# --- Run ---
print(f"\n{'='*60}")
print("Running test-3: percentage extraction")
print(f"{'='*60}")
for image_name, image_path in all_images:
    if output_exists_json(image_name, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
        print(f"⏭ Skipping: {image_name}")
        continue
    benchmark_image_percentages(image_path, image_name, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)
print("\n✅ Test 3 complete.")


Running test-3: percentage extraction
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-3-gn-percentages/001_correct.json
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-3-gn-percentages/001_incorrect.json
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-3-gn-percentages/003_correct.json
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-3-gn-percentages/003_incorrect.json
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-3-gn-percentages/004_correct.json
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-3-gn-percentages/004_incorrect.json
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-3-gn-percentages/005_co

In [14]:
from utils.quanti_benchmarking_3_analysis import run_accuracy_analysis
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-3-gn-percentages"
run_accuracy_analysis(BASE_DIR, EXPERIMENT_NAME, MODEL_NAME)

=== Overall Summary ===


,metric,value
0,pop_accuracy_%,90.0
1,latin_accuracy_%,2.0
2,both_correct_%,2.0
3,pop_accuracy_correct_variant_%,90.0
4,pop_accuracy_incorrect_variant_%,90.0
5,latin_accuracy_correct_variant_%,2.0
6,latin_accuracy_incorrect_variant_%,2.0


=== Per Image Results ===


,image,variant,chart_percentages_pop_pred,chart_percentages_pop_true,chart_percentages_pop_correct,chart_percentages_latin_pred,chart_percentages_latin_true,chart_percentages_latin_correct
0,001_correct,correct,23.5,23.5,True,11.0%,11,False
1,001_incorrect,incorrect,23.5,23.5,True,11.0%,11,False
2,003_correct,correct,23.5,23.5,True,11.0%,11,False
3,003_incorrect,incorrect,23.5,23.5,True,11.0%,11,False
4,004_correct,correct,23.5%,23.5,False,11.0%,11,False
...,...,...,...,...,...,...,...,...
95,096_incorrect,incorrect,23.5,23.5,True,11.0%,11,False
96,098_correct,correct,23.5,23.5,True,11.0%,11,False
97,098_incorrect,incorrect,23.5,23.5,True,11.0%,11,False
98,100_correct,correct,23.5,23.5,True,11.0%,11,False


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-3-gn-percentages/accuracy_scores_test-3-gn-percentages.csv


- Issue here is that quite often the model is providing the % symbol too which immediatly marks the answwer as wrong
- However there is more variation in accuracy for pop than for latin, for latin read the same value regardless of correct/incorrect, wih¿th pop it performed better with incorrect variants...

## old Nr of reactions - cover all reactions - equally distributed (will probably keep out because unrealistic)
Does the overall volume of engagement metrics influence the model's claim verification accuracy, independent of reaction type distribution?

By keeping all reactions equal we are controlling for reaction type bias because the model can't be swayed by seeing mostly angry vs mostly love reactions. 

In [15]:
#!unzip correct/remy-ashford/correct_uniform.zip -d correct/remy-ashford/metrics/uniform
#!unzip incorrect/remy-ashford/incorrect_uniform.zip -d incorrect/remy-ashford/metrics/uniform

The same 50 numbers are reused across all 6 scale values, giving 600 images total per prompt version (50 pairs × 6 scale values × 2 variants)

In [16]:
# codigo viejo incluye two-call approach y se encuentra en copia de este notebook

# 4) Nr of reactions - more realistic -  X total across all reaction types (likes + loves + hahas etc. combined), x being the log numebr so 10, 100, 1000, etc.
Posts were generated under two reaction conditions: uniform, in which all reaction types were set equally, and realistic, in which reactions were distributed using log-scaled weights to approximate empirical engagement patterns on social media.

With uniform reactions, every post at scale_value=1000 showed exactly 1000 likes, 1000 loves, 1000 hahas etc. — which is something that essentially never occurs on real Facebook and could itself be a signal to the VLM that something artificial is happening. The realistic condition removes that artificiality while keeping scale_value as a clean, interpretable independent variable representing **total engagement volume**.

**Why Jitter Matters**
Without jitter, every image index at a given scale_value would produce identical reaction counts. For example, at scale_value=1000 every single one of your 100 images would show:


- Emoji order is always like, love, haha, wow, sad, angry — fixed by the change in the .py file
- Values vary across images because shares are shuffled using seed=i — so image 001 always gets the same distribution but different from image 002
- Total reactions always sum to approximately scale_value — Interpretation 1
- Files land in realistic/ keeping them separate from your uniform/ condition
- Reproducible — rerunning will generate identical files

In [17]:
#!unzip correct/remy-ashford/correct_realistic.zip -d correct/remy-ashford/metrics/realistic
#!unzip incorrect/remy-ashford/incorrect_realistic.zip -d incorrect/remy-ashford/metrics/realistic

In [18]:
import sys
import random
from pathlib import Path
from utils.quanti_benchmarking_4_reactions import (
    PROMPT_VERSIONS, benchmark_image, output_exists_json, ask_question_pixtral
)

# --- Configuration ---
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-4-metrics-realistic-claim-only"
ASK_FN = ask_question_pixtral
SEED = 42
SAMPLE_SIZE = 50
REACTION_VALUES = [10, 100, 1000, 10000, 100000, 1000000]

# --- Build paired sample ---
correct_base = BASE_DIR / "correct/remy-ashford/metrics/realistic"
incorrect_base = BASE_DIR / "incorrect/remy-ashford/metrics/realistic"
sample_dir = correct_base / "10"
all_numbers = sorted([p.name.split("_")[0] for p in sample_dir.glob("*_remy_ashford_c.png")])
print(f"Found {len(all_numbers)} images in {sample_dir}")

random.seed(SEED)
selected_numbers = sorted(random.sample(all_numbers, SAMPLE_SIZE))
print(f"Selected {len(selected_numbers)} numbers: {selected_numbers}")

all_images = []
for scale_value in REACTION_VALUES:
    for num in selected_numbers:
        all_images.append((
            f"{num}_correct_{scale_value}",
            str(correct_base / str(scale_value) / f"{num}_remy_ashford_c.png")
        ))
        all_images.append((
            f"{num}_incorrect_{scale_value}",
            str(incorrect_base / str(scale_value) / f"{num}_remy_ashford_i.png")
        ))
print(f"Total images to process: {len(all_images)}")

# --- Run ---
for prompt_version, prompt_text in PROMPT_VERSIONS.items():
    print(f"\n{'='*60}\nRunning: {prompt_version}\n{'='*60}")
    for image_name, image_path in all_images:
        if output_exists_json(image_name, prompt_version, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME):
            print(f"⏭ Skipping: {image_name} [{prompt_version}]")
            continue
        benchmark_image(image_path, image_name, prompt_version, prompt_text, model, processor, device, BASE_DIR, EXPERIMENT_NAME, MODEL_NAME, ASK_FN)

print("\n✅ All metrics versions complete.")

Found 100 images in /home/jovyan/conformity-llms-facebook-posts/benchmarking/correct/remy-ashford/metrics/realistic/10
Selected 50 numbers: ['001', '003', '004', '005', '006', '007', '012', '014', '015', '017', '018', '020', '021', '023', '025', '026', '028', '029', '030', '032', '036', '039', '044', '047', '052', '054', '055', '058', '059', '063', '065', '068', '069', '070', '072', '076', '078', '080', '082', '083', '085', '087', '089', '090', '091', '094', '095', '096', '098', '100']
Total images to process: 600

Running: v1
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-4-metrics-realistic-claim-only/v1/001_correct_10.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-4-metrics-realistic-claim-only/v1/001_incorrect_10.json [v1]
✅ Saved: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-4-metrics-realistic-claim-only/v1/0

runtime: 35 mins


In [19]:
from utils.quanti_benchmarking_4_analysis import run_accuracy_analysis
BASE_DIR = Path().resolve()
EXPERIMENT_NAME = "test-4-metrics-realistic-claim-only"
run_accuracy_analysis(BASE_DIR, EXPERIMENT_NAME, MODEL_NAME)


Prompt Version : v1
Prompt Text    : Does the text in the post accurately describe the chart? Reply with only 'correct' or 'incorrect'.
=== Overall Summary ===


,metric,value
0,overall_accuracy_%,50.0
1,accuracy_correct_posts_%,100.0
2,accuracy_incorrect_posts_%,0.0


=== Accuracy per Reaction Level ===


/home/jovyan/conformity-llms-facebook-posts/benchmarking/utils/quanti_benchmarking_4_analysis.py:53: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_scale = df.groupby("scale_value").apply(lambda g: pd.Series({


,scale_value,total_images,overall_accuracy_%,correct_posts_accuracy_%,incorrect_posts_accuracy_%
0,10,100.0,50.0,100.0,0.0
1,100,100.0,50.0,100.0,0.0
2,1000,100.0,50.0,100.0,0.0
3,10000,100.0,50.0,100.0,0.0
4,100000,100.0,50.0,100.0,0.0
5,1000000,100.0,50.0,100.0,0.0


...

/home/jovyan/conformity-llms-facebook-posts/benchmarking/utils/quanti_benchmarking_4_analysis.py:53: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_scale = df.groupby("scale_value").apply(lambda g: pd.Series({


,scale_value,total_images,overall_accuracy_%,correct_posts_accuracy_%,incorrect_posts_accuracy_%
0,10,100.0,0.0,0.0,0.0
1,100,100.0,0.0,0.0,0.0
2,1000,100.0,0.0,0.0,0.0
3,10000,100.0,0.0,0.0,0.0
4,100000,100.0,0.0,0.0,0.0
5,1000000,100.0,0.0,0.0,0.0


=== Per Image Results ===


,image,scale_value,variant,ground_truth,prediction,correct
0,001_correct_10,10,correct,correct,the claim states that pop was more popular tha...,False
2,003_correct_10,10,correct,correct,the claim states that pop was more popular tha...,False
4,004_correct_10,10,correct,correct,the claim states that pop was more popular tha...,False
6,005_correct_10,10,correct,correct,"the claim in the text states that ""pop was mor...",False
8,006_correct_10,10,correct,correct,the claim states that pop was more popular tha...,False
...,...,...,...,...,...,...
591,094_incorrect_1000000,1000000,incorrect,incorrect,the text in the post claims that latin was mor...,False
593,095_incorrect_1000000,1000000,incorrect,incorrect,the text in the post claims that latin was mor...,False
595,096_incorrect_1000000,1000000,incorrect,incorrect,the text in the post claims that latin was mor...,False
597,098_incorrect_1000000,1000000,incorrect,incorrect,the claim in the text states that latin was mo...,False


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/benchmarking/outputs/pixtral-12b/quantitative/test-4-metrics-realistic-claim-only/v4/accuracy_scores_v4.csv
